# MIMIC-IV Multi-Target Dataset Exploration

Explore generated pre-tensor `records.csv`, `labels.csv`, and `dataset_metadata.json` files for each target/window dataset. This notebook does not create tensors.

In [ ]:
from pathlib import Path
import json
import os
import sys

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'src' / 'interpretable_ts_vit').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError(f'Could not find repository root from {cwd}')

SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_DIR)

DATA_ROOT = PROJECT_DIR / 'data' / 'mimic_targets'
print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dataset_dirs = sorted(path for path in DATA_ROOT.glob('*/*') if (path / 'records.csv').exists() and (path / 'labels.csv').exists())
print('datasets found:', len(dataset_dirs))
for path in dataset_dirs:
    print(path.relative_to(DATA_ROOT))

In [ ]:
summary_rows = []
for dataset_dir in dataset_dirs:
    records = pd.read_csv(dataset_dir / 'records.csv')
    labels = pd.read_csv(dataset_dir / 'labels.csv')
    metadata = json.loads((dataset_dir / 'dataset_metadata.json').read_text())
    label_counts = labels['label'].value_counts().to_dict()
    summary_rows.append({
        'window': dataset_dir.parent.name,
        'target': dataset_dir.name,
        'patients': len(labels),
        'records': len(records),
        'positive': label_counts.get('true', 0),
        'negative': label_counts.get('false', 0),
        'variables': records['variable'].nunique() if len(records) else 0,
        'target_definition': metadata.get('target_metadata', {}).get('target_definition'),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

In [ ]:
for dataset_dir in dataset_dirs:
    records = pd.read_csv(dataset_dir / 'records.csv')
    labels = pd.read_csv(dataset_dir / 'labels.csv')
    title = f'{dataset_dir.parent.name} / {dataset_dir.name}'
    print('\n' + title)
    display(labels['label'].value_counts().rename('patients').to_frame())
    if records.empty:
        print('No records')
        continue
    display(records['variable'].value_counts().rename('records').to_frame())
    coverage = records.groupby('variable')['patient_id'].nunique().sort_values(ascending=False).rename('patients_with_variable').to_frame()
    coverage['coverage_fraction'] = coverage['patients_with_variable'] / max(1, labels['patient_id'].nunique())
    display(coverage)
    numeric = records.copy()
    numeric['value'] = pd.to_numeric(numeric['value'], errors='coerce')
    display(numeric.groupby('variable')['value'].describe()[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

In [ ]:
for dataset_dir in dataset_dirs:
    records = pd.read_csv(dataset_dir / 'records.csv')
    if records.empty:
        continue
    records['timestamp'] = pd.to_datetime(records['timestamp'])
    records['hours_since_admission'] = (records['timestamp'] - pd.Timestamp('2000-01-01')) / pd.Timedelta(hours=1)
    ax = records['hours_since_admission'].hist(bins=48, figsize=(8, 3))
    ax.set_title(f'{dataset_dir.parent.name} / {dataset_dir.name}: event time distribution')
    ax.set_xlabel('hours since admission')
    ax.set_ylabel('records')
    plt.show()

In [ ]:
for dataset_dir in dataset_dirs:
    records = pd.read_csv(dataset_dir / 'records.csv')
    labels = pd.read_csv(dataset_dir / 'labels.csv')
    if records.empty:
        continue
    frame = records.merge(labels, on='patient_id', how='inner')
    frame['value'] = pd.to_numeric(frame['value'], errors='coerce')
    comparison = frame.groupby(['variable', 'label'])['value'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index()
    print('\n' + f'{dataset_dir.parent.name} / {dataset_dir.name}: positive-negative variable comparison')
    display(comparison)